In [ ]:
print('Importing relevant modules ...')
import warnings
warnings.filterwarnings('ignore',category=RuntimeWarning)
import os
import numpy as np
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pylab import rcParams
import matplotlib
from matplotlib import rc;rc('text', usetex=True);rc('font', weight='bold');matplotlib.rcParams['text.latex.preamble'] = r'\boldmath'
rcParams['font.family'] = 'serif'
rc('text.latex',preamble=r'\usepackage{/Users/kevinlevy/Documents/cluster_lensing/files/apjfonts}')
from mpl_toolkits.axes_grid1 import make_axes_locatable
color_arr = ['olivedrab', 'steelblue', 'goldenrod']
import cosmo
import lensing
import lensing_estimator
import mockobs
import stats
import utils

In [ ]:
print('Defining relevant parameters ...')
nber_clus = 2500
nber_rand = 50000
cutout_size_am = 6
cutout_size_for_grad_est_am = 6
l_cut = 2000
average = 1
average_run = 1
nber_runs = 20
nx = 120
ny = 120
dx = 0.5
dy = 0.5
reso_arcmin = 0.5
map_params = [nx, dx, ny, dy]
beam_fwhm = 1.0 # arcmin
noiseval_white = 2.0 # uK-arcmin
l = np.arange(10001)
cl = cosmo.cmb_power_spectra(l)['TT']
bl = mockobs.beam_power_spectrum(beam_fwhm, l)
nl = mockobs.instrumental_noise_power_spectrum(noiseval_white, l)
cl_noise = mockobs.instrumental_noise_power_spectrum(noiseval_white, l, beam_fwhm)

z = 0.7
mass_reso = 0.01
mass_min = 0.0
mass_max = 2.5
mass_int = np.arange(mass_min, mass_max, mass_reso)
M_input = 2e14
offsets = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]

# for plotting
xmin, xmax = -nx*reso_arcmin/2, nx*reso_arcmin/2
ymin, ymax = -ny*reso_arcmin/2, ny*reso_arcmin/2 
extent_arcmin = [xmin, xmax, ymin, ymax]
extent_degrees = np.asarray(extent_arcmin)/60
fontsize = 19
labelsize = 15

    
save_loc_results = '/Users/kevinlevy/Documents/cluster_lensing/results/ref2_alt'
if not os.path.exists(save_loc_results):
    os.makedirs(save_loc_results)

In [ ]:
print('Checking likelhihood curves ...')
L_arr_biased = np.load(save_loc_results+'/likelihoods_biased.npy', allow_pickle=True)
L_ip_arr_biased = np.load(save_loc_results+'/likelihoods_finer_biased.npy', allow_pickle=True)
nber_runs = 10
s,e = 0, 10
L_arr_biased = [L_arr_biased[0][s:e], L_arr_biased[1][s:e], L_arr_biased[2][s:e],
         L_arr_biased[3][s:e], L_arr_biased[4][s:e], L_arr_biased[5][s:e],
         L_arr_biased[6][s:e], L_arr_biased[7][s:e], L_arr_biased[8][s:e]]
L_ip_arr_biased = [L_ip_arr_biased[0][s:e], L_ip_arr_biased[1][s:e], L_ip_arr_biased[2][s:e],
                   L_ip_arr_biased[3][s:e], L_ip_arr_biased[4][s:e], L_ip_arr_biased[5][s:e],
                   L_ip_arr_biased[6][s:e], L_ip_arr_biased[7][s:e], L_ip_arr_biased[8][s:e]]
comb_L_arr_biased = []
comb_L_ip_arr_biased = []
comb_median_mass_arr_biased = []
comb_error_arr_biased = []
for i in range(len(offsets)):   
    M_ip_comb_biased, L_ip_comb_biased, median_value_comb_finer_biased, error_comb_finer_biased = stats.combined_likelihood(mass_int, L_arr_biased[i], normalize = True, finer_reso = True)
    print(median_value_comb_finer_biased, error_comb_finer_biased) 
    comb_L_ip_arr_biased.append(L_ip_comb_biased)
    comb_median_mass_arr_biased.append(median_value_comb_finer_biased)
    comb_error_arr_biased.append(error_comb_finer_biased)
combined_median_masses_results_file_biased = ''
for i in range(len(comb_median_mass_arr_biased)):
    combined_median_masses_results_file_biased += "{0:11.2f}{1:>1}{2:5.2f}".format(comb_median_mass_arr_biased[i], "+-", comb_error_arr_biased[i]) + "\n"  
file_biased = open(save_loc_results+'/mass_results_biased.txt',"w")
file_biased.write(combined_median_masses_results_file_biased)
file_biased.close()


print('Checking likelhihood curves ...')
L_arr= np.load(save_loc_results+'/likelihoods.npy', allow_pickle=True)
L_ip_arr = np.load(save_loc_results+'/likelihoods_finer.npy', allow_pickle=True)
nber_runs = 10
s,e = 0, 10
L_arr = [L_arr[0][s:e], L_arr[1][s:e], L_arr[2][s:e],
         L_arr[3][s:e], L_arr[4][s:e], L_arr[5][s:e],
         L_arr[6][s:e], L_arr[7][s:e], L_arr[8][s:e]]
L_ip_arr = [L_ip_arr[0][s:e], L_ip_arr[1][s:e], L_ip_arr[2][s:e],
            L_ip_arr[3][s:e], L_ip_arr[4][s:e], L_ip_arr[5][s:e],
            L_ip_arr[6][s:e], L_ip_arr[7][s:e], L_ip_arr[8][s:e]]
comb_L_arr = []
comb_L_ip_arr = []
comb_median_mass_arr = []
comb_error_arr = []
for i in range(len(offsets)):   
    M_ip_comb, L_ip_comb, median_value_comb_finer, error_comb_finer = stats.combined_likelihood(mass_int, L_arr[i], normalize = True, finer_reso = True)
    print(median_value_comb_finer, error_comb_finer) 
    comb_L_ip_arr.append(L_ip_comb)
    comb_median_mass_arr.append(median_value_comb_finer)
    comb_error_arr.append(error_comb_finer)
combined_median_masses_results_file = ''
for i in range(len(comb_median_mass_arr)):
    combined_median_masses_results_file += "{0:11.2f}{1:>1}{2:5.2f}".format(comb_median_mass_arr[i], "+-", comb_error_arr[i]) + "\n"  
file = open(save_loc_results+'/mass_results.txt',"w")
file.write(combined_median_masses_results_file)
file.close()


In [ ]:
fontsize = 17
labelsize = 15
legendsize = 15

In [ ]:
comb_mass_arr_biased = [comb_median_mass_arr_biased[0], comb_median_mass_arr_biased[1], 
                        comb_median_mass_arr_biased[2], comb_median_mass_arr_biased[3],
                        comb_median_mass_arr_biased[4], comb_median_mass_arr_biased[5],         
                        comb_median_mass_arr_biased[6], comb_median_mass_arr_biased[7],
                       comb_median_mass_arr_biased[8]]
comb_mass_arr = [comb_median_mass_arr[0], comb_median_mass_arr[1], 
                 comb_median_mass_arr[2], comb_median_mass_arr[3],
                 comb_median_mass_arr[4], comb_median_mass_arr[5],         
                 comb_median_mass_arr[6], comb_median_mass_arr[7],
                comb_median_mass_arr[8]]
comb_mass_error_arr_biased = [comb_error_arr_biased[0], comb_error_arr_biased[1], 
                              comb_error_arr_biased[2], comb_error_arr_biased[3],
                              comb_error_arr_biased[4], comb_error_arr_biased[5],
                              comb_error_arr_biased[6], comb_error_arr_biased[7],
                              comb_error_arr_biased[8]]
comb_mass_error_arr = [comb_error_arr[0], comb_error_arr[1], 
                       comb_error_arr[2], comb_error_arr[3],
                       comb_error_arr[4], comb_error_arr[5],
                       comb_error_arr[6], comb_error_arr[7],
                       comb_error_arr[8]]



print(offsets)
print((np.asarray(comb_mass_arr_biased)-M_input*1e-14)/(M_input*1e-14))
print(np.asarray(comb_mass_error_arr_biased)/(M_input*1e-14))

print((np.asarray(comb_mass_arr)-M_input*1e-14)/(M_input*1e-14))
print(np.asarray(comb_mass_error_arr)/(M_input*1e-14))

color_arr = ['lightseagreen', 'darkorchid', 'goldenrod']
fig, ax = plt.subplots(figsize=(6,4))
ax.errorbar(offsets, (np.asarray(comb_mass_arr_biased)-M_input*1e-14)/(M_input*1e-14), 
            np.asarray(comb_mass_error_arr_biased)/(M_input*1e-14), color = color_arr[0], 
            marker = 's', markersize = 6., elinewidth = 2.5, ls = '', label = 'Biased', capsize=2.5, capthick=2.0)
ax.errorbar(offsets, (np.asarray(comb_mass_arr)-M_input*1e-14)/(M_input*1e-14), 
            np.asarray(comb_mass_error_arr)/(M_input*1e-14), color = color_arr[1], 
            marker = 'o', markersize = 6., elinewidth = 2.5, ls = '', label = 'Mitigated', capsize=2.5, capthick=2.0)
ax.tick_params(labelsize = labelsize, length = 5.5, width = 1.5)
ax.axhline(0, color = 'black') 
ax.set_xlabel(r'$\sigma_{\rm offset}$ [arcmin] ', fontsize = fontsize)
ax.set_ylabel(r'$\frac{M_{\rm fit}}{M_{\rm input}}-1$', fontsize = fontsize)
ax.legend(loc = 'lower left', prop={'size': legendsize}) 
ax.axvline(0.5, ls = '--', color = 'black')
ax.set_ylim([-1.05, 0.25])
ax.set_yticks([-1., -0.75, -0.5, -0.25, 0.0, 0.2])
plt.show()
fig.savefig(save_loc_results+'/cluster_positions.pdf', dpi = 200., bbox_inches = 'tight', pad_inches = 0.1)


In [ ]:
[0.00 0.25 0.50 0.75 1.00 1.25 1.50 1.75 2.00]
[0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01]
[0.01 0.01 0.01 0.01 0.01 0.02 0.02 0.03 0.03]

In [ ]:
0.0091147/0.0083969

In [ ]:
0.03078067/0.0063969

In [ ]:
print(np.asarray(comb_mass_error_arr)/np.asarray(comb_mass_error_arr_biased))